# RAG evaluation

This notebook displays the 25-question benchmark for the five PDFs in `data/`. The scripts use the parser and prompt from `eda_workbook.ipynb` plus `ingest.py` and `retrieval.py`. Labels, metric definitions, and limitations are in [evaluation/README.md](../evaluation/README.md).

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown

root = Path.cwd() if (Path.cwd() / "evaluation").is_dir() else Path.cwd().parent
assert (root / "evaluation/benchmark.json").exists(), "Run from the repo root or notebooks/"
print(root)

# Change run_name when changing model, prompt, corpus or retrieval settings.
run_name = "ordered_frozen_v1"
run_dir = root / "evaluation/results" / run_name
retrieval_path = run_dir / "retrieval.json"
answers_path = run_dir / "answers.json"
checks_path = run_dir / "answer_checks.json"

## Retrieval

Run this cell after `python -m pip install -r evaluation/requirements.txt`. The first run downloads models and builds the Chroma index. The command uses the current assistant retrieval settings and saves exact passages. An existing snapshot is reused; change `run_name` to create a new retrieval experiment.

In [ ]:
if not retrieval_path.exists():
    subprocess.run([sys.executable, str(root / "evaluation/run_retrieval.py"),
                    "--output", str(retrieval_path), "--per-paper", "4", "--fetch-k", "60",
                    "--rerank-model", "cross-encoder/ms-marco-MiniLM-L6-v2",
                    "--max-rerank-gap", "8", "--order-by-vector"], cwd=root, check=True)
else:
    print(f"Reusing frozen passages: {retrieval_path}")

In [ ]:
retrieval = json.loads(retrieval_path.read_text())
display(pd.DataFrame([retrieval["summary"]]).T.rename(columns={0: "value"}))
rows = retrieval["queries"]
display(pd.DataFrame([{"id": q["id"], "query": q["query"], "gold": ", ".join(q["relevant_papers"]), "retrieved": ", ".join(q["retrieved_papers"]), "evidence": f"{sum(q['evidence_hits'])}/{len(q['evidence_hits'])}"} for q in rows]))

## Fresh answers

Start Ollama and pull `llama3.2` in a terminal (`ollama serve`, then `ollama pull llama3.2`). Answers use the frozen passages without retrieving again. Compatible runs resume; changes to the run configuration require a new answer output path. To compare models or prompts on identical evidence, keep `retrieval_path` and change `answers_path` and `checks_path`. Saved outputs in `eda_workbook.ipynb` came from another corpus and are not scored here.

In [ ]:
subprocess.run([sys.executable, str(root / "evaluation/run_answers.py"),
                "--retrieval", str(retrieval_path), "--output", str(answers_path)], cwd=root, check=True)

In [ ]:
subprocess.run([sys.executable, str(root / "evaluation/check_answers.py"),
                "--answers", str(answers_path), "--output", str(checks_path)], cwd=root, check=True)
checks = json.loads(checks_path.read_text())
print(checks["validation_scope"])
display(pd.DataFrame([checks["summary"]]).T.rename(columns={0: "value"}))

In [ ]:
answers = json.loads(answers_path.read_text())["answers"]
selected_id = "q01"  # change to any ID from q01 to q25
row = next(a for a in answers if a["id"] == selected_id)
display(Markdown(f"### {selected_id}: {row['query']}\n\n{row['answer']}\n\n**Sources**\n\n{row['source_list']}"))
for passage in row["passages"]:
    display(Markdown(f"**{passage['paper_id']} · {passage['section']}**\n\n> {passage['text'][:1000]}..."))

## Review rubric

For each answer, check that its citations point to supporting passages, all material claims are correct, each requested fact is covered or explicitly missing, and the answer contains a direct quote when required by issue #5. For q24–q25, check that the model declines to invent KPMG or market data. Record the manual judgments separately from retrieval metrics; the presence of a citation does not establish factual accuracy.

The completed per-query review is in [evaluation/results/answer_review.md](../evaluation/results/answer_review.md), and improvement opportunities are in [evaluation/results/findings.md](../evaluation/results/findings.md).

## BEIR SciFact relevance test

This separate run downloads the [official SciFact archive](https://github.com/beir-cellar/beir), selects 20 labeled test claims, and searches 500 articles including all judged relevant ones. It is a small diagnostic subset, not a published full-corpus BEIR score. SciFact labels retrieval relevance only.

In [ ]:
subprocess.run([sys.executable, str(root / "evaluation/run_beir_scifact.py")], cwd=root, check=True)

In [ ]:
beir = json.loads((root / "evaluation/results/beir_scifact_20q_500d.json").read_text())
display(pd.DataFrame([beir["summary"]]).T.rename(columns={0: "value"}))
display(pd.DataFrame([{"query_id": q["query_id"], "claim": q["query"], "relevant": ", ".join(q["relevant_ids"]), "retrieved": ", ".join(q["retrieved_ids"][:5]), "recall@5": q["metrics"]["recall_at_5"]} for q in beir["queries"]]))